# گام پنجم: آموزش نیمه‌نظارتی ParsBERT در Colab
این نوت‌بوک مدل supervised موجود را بارگذاری می‌کند، روی مجموعهٔ مصنوعی برچسب پیشنهادی می‌سازد،
پیش‌بینی‌های با احتمال حداقل **۰٫۹۰** را می‌پذیرد و یک کپی از مدل را با آن‌ها ادامهٔ آموزش می‌دهد.
ورودی فقط **body** است. برچسب شبه‌سازی‌شده فقط برای **recommendation_status** استفاده می‌شود.

**بودجهٔ اجرا:** حداکثر یک epoch اضافه، همان دو میلیون نمونهٔ اصلی، batch مؤثر ۳۲.
ارزیابی نهایی برای هر دو مدل روی کل validation و test انجام می‌شود.
نتیجهٔ بهتر تضمین‌شده نیست؛ افت عملکرد هم نتیجهٔ معتبر آزمایش است.

سلول‌ها را به ترتیب اجرا کنید. پس از قطع اتصال، دوباره از بالا اجرا کنید؛ بخش‌های کامل‌شده از Drive خوانده می‌شوند.
برای تغییر تنظیمات آزمایش، نام پوشهٔ خروجی را تغییر دهید؛ خروجی دو آزمایش را مخلوط نکنید.


## ۰. فایل‌ها را کجا قرار بدهم؟
از همان پوشهٔ قبلی در MyDrive استفاده کنید. ساختار لازم:

    digikala-transformer/
      notebooks/
        04_semi_supervised_colab.ipynb
      data/clean_data/split_body_cleaned/
        train_supervised.csv
        train_pseudo_unlabeled.csv
        validation.csv
        test.csv
        split_manifest.json
        mask_manifest.csv
      outputs/parsbert_finetuning/artifacts/
        model.safetensors
        model_metadata.json
        tokenizer/
          تمام فایل‌های tokenizer قبلی
      outputs/semi_supervised_v1/
        این پوشه و زیرپوشه‌هایش خودکار ساخته می‌شوند

اگر مدل در Drive قبلی نیست، محتوای پوشهٔ محلی models/parsbert_finetuning/artifacts را
در outputs/parsbert_finetuning/artifacts بارگذاری کنید.
فایل‌های داده را از data/clean_data/split_body_cleaned بردارید؛ manifest قدیمی داخل reports را استفاده نکنید.
natural_unlabeled و دادهٔ خام لازم نیستند. mask_manifest فقط در بخش آخر خوانده می‌شود.

نوت‌بوک را در پوشهٔ notebooks در Drive نگه دارید و با Colab باز کنید؛
یا در Colab از File > Upload notebook آن را بارگذاری کنید و نسخه‌ای در Drive ذخیره کنید.
از Runtime > Change runtime type یک GPU انتخاب کنید.
نوت‌بوک از وزن supervised موجود استفاده می‌کند؛ دانلود دوبارهٔ وزن پایه لازم نیست.


## ۱. نصب کتابخانه‌ها و اتصال Drive
نسخهٔ Transformers با اجرای قبلی یکسان است. PyTorch نصب‌شدهٔ Colab را تغییر نمی‌دهیم.
اگر Colab بعد از نصب درخواست restart کرد، restart کنید و از سلول بعد ادامه دهید.


In [ ]:
%pip -q install "transformers==4.57.6" safetensors scikit-learn pandas matplotlib tqdm


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, math, time, random, shutil, hashlib, gc
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from safetensors.torch import load_file, save_file
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROJECT = Path("/content/drive/MyDrive/digikala-transformer")
DATA = PROJECT / "data/clean_data/split_body_cleaned"
BASE = PROJECT / "outputs/parsbert_finetuning/artifacts"
OUT = PROJECT / "outputs/semi_supervised_v1"
LOCAL = Path("/content/digikala_semi_cache")
for folder in [LOCAL, OUT, OUT/"cache", OUT/"pseudo", OUT/"checkpoints", OUT/"reports", OUT/"artifacts"]:
    folder.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("GPU فعال نیست؛ Runtime > Change runtime type را بررسی کنید.")
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))


## ۲. تنظیمات ثابت و شناسایی فایل‌ها
بهتر است این تنظیمات را برای اجرای اصلی تغییر ندهید.
آموزش فقط روی یک epoch اضافه محدود شده است. هر ۱۰ هزار گام، validation پایش می‌شود؛
پس از دو پایش متوالی بدون بهبود متوقف می‌شویم. بهترین مدل **آموزش‌دیده در این آزمایش** نگه داشته می‌شود،
حتی اگر از supervised ضعیف‌تر باشد.

هر ۲ هزار گام وضعیت آموزش ذخیره می‌شود. هر جلسهٔ آموزش حدود ۱۲۰ دقیقه اجرا می‌شود و سپس
در مرز یک گام کامل ذخیره و مکث می‌کند. این مکث پایان آزمایش نیست؛ دوباره سلول آموزش را اجرا کنید.
این زمان حد نرم است؛ ذخیره و validation می‌توانند آن را کمی طولانی‌تر کنند.


In [ ]:
SEED = 2026
THRESHOLD = 0.90
MAX_LENGTH = 40
BATCH_SIZE = 8
GRAD_ACCUMULATION = 4
INFERENCE_BATCH = 64
EPOCHS = 1
ENCODER_LR, HEAD_LR = 2e-5, 1e-4
WEIGHT_DECAY, WARMUP_RATIO = 0.01, 0.06
CHUNK_ROWS = 20_000
MONITOR_SIZE = 100_000
SAVE_EVERY = 2_000
EVAL_EVERY = 10_000
PATIENCE = 2
SESSION_MINUTES = 120  # مکث هر جلسه؛ قابل تغییر بدون تغییر تعریف آزمایش
assert THRESHOLD == 0.90 and EPOCHS == 1
TARGETS = {"recommendation_status": ["recommended", "not_recommended", "no_idea"],
           "rating_band": ["low", "medium", "high"]}
IGNORE = -100
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, value):
    path = Path(path)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temp, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(4*1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()

def publish(local_path, drive_path):
    # فایل ناتمام نام نهایی نمی‌گیرد.
    temp = Path(str(drive_path) + ".tmp")
    shutil.copyfile(local_path, temp)
    os.replace(temp, drive_path)

manifest = read_json(DATA/"split_manifest.json")
metadata = read_json(BASE/"model_metadata.json")
previous = metadata["training_config"]
assert metadata["targets"] == TARGETS
assert metadata["max_length"] == MAX_LENGTH
assert manifest["rating_band_thresholds"] == metadata["rating_band_thresholds"]
assert manifest["split_seed"] == previous["split_seed"] == SEED
assert manifest["mask_seed"] == "digikala-semi-supervised-v1"
assert manifest["mask_rate"] == 0.20
assert previous["train_sampling"] == "seeded uniform sample by source row index"
TRAIN_SIZE = int(previous["train_rows"])
assert TRAIN_SIZE == 2_000_000 and previous["train_sample_seed"] == SEED
assert previous["batch_size"] == BATCH_SIZE and previous["gradient_accumulation"] == GRAD_ACCUMULATION
assert previous["encoder_lr"] == ENCODER_LR and previous["head_lr"] == HEAD_LR
assert previous["split_strategy"] == manifest["split_strategy"]["method"]

# mask_manifest تا پایان آموزش خوانده نمی‌شود.
files = ["train_supervised.csv", "train_pseudo_unlabeled.csv", "validation.csv", "test.csv"]
fingerprints = {}
for name in tqdm(files, desc="Checking/copying inputs"):
    source, dest = DATA/name, LOCAL/name
    source_hash = sha256(source)
    if not dest.exists() or sha256(dest) != source_hash:
        publish(source, dest)
    if sha256(dest) != source_hash:
        raise ValueError("کپی فایل ناقص است: " + name)
    fingerprints[name] = source_hash
fingerprints["split_manifest.json"] = sha256(DATA/"split_manifest.json")
fingerprints["model.safetensors"] = sha256(BASE/"model.safetensors")
fingerprints["model_metadata.json"] = sha256(BASE/"model_metadata.json")
fingerprints["tokenizer"] = {p.name: sha256(p) for p in sorted((BASE/"tokenizer").iterdir()) if p.is_file()}

assert list(pd.read_csv(LOCAL/"train_pseudo_unlabeled.csv", nrows=0).columns) == ["id", "body"]
settings = dict(seed=SEED, threshold=THRESHOLD, train_size=TRAIN_SIZE, max_length=MAX_LENGTH,
                batch_size=BATCH_SIZE, accumulation=GRAD_ACCUMULATION, inference_batch=INFERENCE_BATCH,
                epochs=EPOCHS, encoder_lr=ENCODER_LR, head_lr=HEAD_LR,
                weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO, chunk_rows=CHUNK_ROWS,
                monitor_size=MONITOR_SIZE, eval_every=EVAL_EVERY, patience=PATIENCE)
run = {"settings": settings, "inputs": fingerprints}
run_path = OUT/"run_identity.json"
if run_path.exists():
    if read_json(run_path) != run:
        raise ValueError("فایل‌ها یا تنظیمات عوض شده‌اند؛ OUT را به پوشهٔ آزمایش جدید تغییر دهید.")
else:
    write_json(run_path, run)
RUN_ID = hashlib.sha256(json.dumps(run, sort_keys=True).encode()).hexdigest()
print("Supervised sample:", TRAIN_SIZE)
print("Pseudo pool:", manifest["output_row_counts"]["train_pseudo_unlabeled"])


## ۳. بارگذاری مدل supervised
معماری دو سر دقیقاً مانند نوت‌بوک قبلی است.
فقط پیکربندی encoder، در اولین اجرا، از Hugging Face دریافت می‌شود؛ وزن‌ها از فایل شما می‌آیند.
تمام پارامترهای encoder و هر دو سر قابل آموزش هستند.


In [ ]:
config_path = OUT/"cache/encoder_config.json"
if not config_path.exists():
    config = AutoConfig.from_pretrained(metadata["base_checkpoint"])
    config.to_json_file(str(config_path))
config = AutoConfig.from_pretrained(str(config_path))
tokenizer = AutoTokenizer.from_pretrained(str(BASE/"tokenizer"), local_files_only=True, use_fast=True)

class TwoHeadParsBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_config(config)
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({t: nn.Linear(config.hidden_size, len(labels)) for t, labels in TARGETS.items()})

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled = self.dropout(out.last_hidden_state[:, 0])
        return {t: head(pooled) for t, head in self.heads.items()}

model = TwoHeadParsBERT()
model.load_state_dict(load_file(str(BASE/"model.safetensors"), device="cpu"), strict=True)
model.to(DEVICE).eval()
model.encoder.gradient_checkpointing_enable()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total:", f"{total_params:,}", "| Trainable:", f"{trainable_params:,}")
write_json(OUT/"reports/environment.json", {
    "torch": torch.__version__, "transformers": __import__("transformers").__version__,
    "numpy": np.__version__, "pandas": pd.__version__, "sklearn": __import__("sklearn").__version__,
    "gpu": torch.cuda.get_device_name(0), "total_parameters": total_params,
    "trainable_parameters": trainable_params, "encoder_config_sha256": sha256(config_path),
    "early_stopping": "two non-improving validation checks; max one additional epoch",
    "selection": "highest mean Macro F1 of both heads among semi-supervised validation checks",
    "differences_from_supervised": ["one additional epoch maximum", "validation every 10000 optimizer steps",
                                   "inference batch 64; training effective batch stays 32"],
})

def predict(frame):
    predictions = {t: [] for t in TARGETS}
    confidence = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(frame), INFERENCE_BATCH):
            texts = frame["body"].iloc[start:start+INFERENCE_BATCH].fillna("").astype(str).tolist()
            encoded = tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt")
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(encoded["input_ids"].to(DEVICE), encoded["attention_mask"].to(DEVICE))
            for t in TARGETS:
                predictions[t].extend(logits[t].argmax(-1).cpu().tolist())
            confidence.extend(logits["recommendation_status"].float().softmax(-1).max(-1).values.cpu().tolist())
    return {t: np.asarray(v, dtype=np.int64) for t, v in predictions.items()}, np.asarray(confidence)


## ۴. ساخت pseudo-label با آستانهٔ ثابت ۰٫۹۰
این مرحله فقط id و body را می‌خواند. هر ۲۰ هزار نظر، نتیجه در Drive ذخیره می‌شود.
بعد از قطع اتصال، بخش‌های ذخیره‌شده تکرار نمی‌شوند.
pool_row شمارهٔ ردیف در فایل بدون برچسب است؛ حتی اگر id تکراری باشد، ردیف‌ها با هم اشتباه نمی‌شوند.
مدل teacher ثابت می‌ماند و در این مرحله آموزش نمی‌بیند.


In [ ]:
model.load_state_dict(load_file(str(BASE/"model.safetensors"), device="cpu"), strict=True)
accepted_count = 0
class_counts = np.zeros(3, dtype=np.int64)
pool_count = 0
started = time.time()
new_rows = 0
reader = pd.read_csv(LOCAL/"train_pseudo_unlabeled.csv", dtype=str, keep_default_na=False, chunksize=CHUNK_ROWS)
for part, frame in enumerate(tqdm(reader, desc="Pseudo-labels")):
    dest = OUT/"pseudo"/f"part_{part:05d}.csv"
    if dest.exists():
        result = pd.read_csv(dest, dtype={"id":str}, keep_default_na=False)
    else:
        pred, confidence = predict(frame)
        result = pd.DataFrame({"pool_row": np.arange(pool_count, pool_count+len(frame)),
                               "id": frame["id"].to_numpy(),
                               "prediction": pred["recommendation_status"],
                               "confidence": confidence})
        temp = LOCAL/"pseudo_part.csv"
        result.to_csv(temp, index=False)
        publish(temp, dest)
        new_rows += len(frame)
    assert np.array_equal(result["id"].to_numpy(), frame["id"].to_numpy())
    assert np.array_equal(result["pool_row"].to_numpy(), np.arange(pool_count, pool_count+len(frame)))
    assert result["confidence"].between(0, 1).all()
    assert result["prediction"].isin(range(3)).all()
    accepted = result["confidence"].ge(THRESHOLD)
    accepted_count += int(accepted.sum())
    class_counts += np.bincount(result.loc[accepted, "prediction"], minlength=3)
    pool_count += len(frame)
    if new_rows and part % 10 == 0:
        speed = new_rows / max(time.time()-started, 1)
        remaining = (manifest["output_row_counts"]["train_pseudo_unlabeled"]-pool_count)/speed/60
        print(f"Accepted: {accepted_count:,} | remaining inference ~{remaining:.1f} min")

assert pool_count == manifest["output_row_counts"]["train_pseudo_unlabeled"]
pseudo_summary = {"pool_rows": pool_count, "accepted_rows": accepted_count,
                  "accepted_percent": 100*accepted_count/max(pool_count,1),
                  "threshold": THRESHOLD,
                  "class_counts": dict(zip(TARGETS["recommendation_status"], class_counts.tolist())),
                  "prediction_hashes": {p.name: sha256(p) for p in sorted((OUT/"pseudo").glob("part_*.csv"))}}
summary_path = OUT/"reports/pseudo_selection.json"
if summary_path.exists():
    assert read_json(summary_path) == pseudo_summary, "فایل‌های pseudo بعد از انتخاب تغییر کرده‌اند."
else:
    write_json(summary_path, pseudo_summary)
display(pd.DataFrame({"class": TARGETS["recommendation_status"], "accepted": class_counts}))
print("Accepted:", accepted_count, "/", pool_count, f"({pseudo_summary['accepted_percent']:.2f}%)")
if accepted_count == 0:
    raise RuntimeError("هیچ pseudo-label پذیرفته نشد؛ آستانه را تغییر ندهید و همین نتیجه را گزارش کنید.")


## ۵. آماده‌کردن دادهٔ آموزشی، فقط یک بار
دو میلیون ردیف اصلی با **همان seed و روش نمونه‌گیری قبلی** انتخاب می‌شوند.
متن‌ها یک بار توکن می‌شوند و در فایل‌های NumPy ذخیره می‌شوند؛ در زمان آموزش دیگر tokenizer اجرا نمی‌شود.
فایل‌ها با mmap خوانده می‌شوند: لازم نیست همهٔ توکن‌ها هم‌زمان در RAM باشند.

برچسب ناموجود برابر منفی ۱۰۰ است. برای همهٔ pseudo-labelها، برچسب rating_band منفی ۱۰۰ می‌شود.
در آموزش، نمونه‌های اصلی و pseudo-labelها با هم به‌صورت تصادفی مخلوط می‌شوند.

این آماده‌سازی CPUمحور است و GPU آموزش نمی‌بیند. اگر وسط آماده‌سازی قطع شود،
فقط همین آماده‌سازی از ابتدا تکرار می‌شود؛ پیش‌بینی‌های مرحلهٔ قبل باقی می‌مانند.
پس از ذخیرهٔ کامل cache، در جلسه‌های بعد از آن استفاده می‌شود.


In [ ]:
LABEL_TO_ID = {t: {label:i for i,label in enumerate(names)} for t,names in TARGETS.items()}
TRAIN_TOTAL = TRAIN_SIZE + accepted_count
cache_info_path = OUT/"cache/token_cache.json"

def target_array(frame):
    return np.column_stack([
        frame[t].astype("string").str.strip().map(LABEL_TO_ID[t]).fillna(IGNORE).to_numpy(dtype=np.int64)
        for t in TARGETS
    ])

def supervised_parts():
    total = int(manifest["output_row_counts"]["train_supervised"])
    selected = np.sort(np.random.default_rng(SEED).choice(total, size=TRAIN_SIZE, replace=False))
    offset = 0
    for frame in pd.read_csv(LOCAL/"train_supervised.csv", usecols=["body", *TARGETS], chunksize=CHUNK_ROWS):
        left, right = np.searchsorted(selected, [offset, offset+len(frame)])
        chosen = frame.iloc[selected[left:right]-offset]
        offset += len(frame)
        if len(chosen):
            yield chosen
    assert offset == total

def pseudo_parts():
    offset = 0
    reader = pd.read_csv(LOCAL/"train_pseudo_unlabeled.csv", dtype=str, keep_default_na=False, chunksize=CHUNK_ROWS)
    for part, frame in enumerate(reader):
        result = pd.read_csv(OUT/"pseudo"/f"part_{part:05d}.csv", dtype={"id":str}, keep_default_na=False)
        assert np.array_equal(result["id"].to_numpy(), frame["id"].to_numpy())
        assert np.array_equal(result["pool_row"].to_numpy(), np.arange(offset, offset+len(frame)))
        offset += len(frame)
        keep = result["confidence"].ge(THRESHOLD).to_numpy()
        chosen = frame.loc[keep, ["body"]].copy()
        chosen["recommendation_status"] = [TARGETS["recommendation_status"][i] for i in result.loc[keep, "prediction"]]
        chosen["rating_band"] = pd.NA
        if len(chosen):
            yield chosen

cache_names = ["train_tokens.npy", "train_labels.npy"]
if not cache_info_path.exists():
    ids_out = np.lib.format.open_memmap(LOCAL/cache_names[0], mode="w+", dtype=np.int32,
                                      shape=(TRAIN_TOTAL, MAX_LENGTH))
    labels_out = np.lib.format.open_memmap(LOCAL/cache_names[1], mode="w+", dtype=np.int64,
                                         shape=(TRAIN_TOTAL, len(TARGETS)))
    position = 0
    for source_name, parts in [("supervised", supervised_parts()), ("pseudo", pseudo_parts())]:
        for frame in tqdm(parts, desc="Tokenize "+source_name):
            # توکن‌سازی در بسته‌های کوچک برای کنترل RAM
            for start in range(0, len(frame), 1024):
                small = frame.iloc[start:start+1024]
                encoded = tokenizer(small["body"].fillna("").astype(str).tolist(),
                                    max_length=MAX_LENGTH, truncation=True,
                                    padding="max_length", return_attention_mask=False,
                                    return_token_type_ids=False)
                end = position + len(small)
                ids_out[position:end] = np.asarray(encoded["input_ids"], dtype=np.int32)
                labels_out[position:end] = target_array(small)
                position = end
    assert position == TRAIN_TOTAL
    ids_out.flush(); labels_out.flush()
    del ids_out, labels_out
    cache_hashes = {}
    for name in cache_names:
        publish(LOCAL/name, OUT/"cache"/name)
        cache_hashes[name] = sha256(LOCAL/name)
    write_json(cache_info_path, {"run_id":RUN_ID, "rows":TRAIN_TOTAL, "hashes":cache_hashes})

cache_info = read_json(cache_info_path)
assert cache_info["run_id"] == RUN_ID and cache_info["rows"] == TRAIN_TOTAL
for name in cache_names:
    if not (LOCAL/name).exists() or sha256(LOCAL/name) != cache_info["hashes"][name]:
        publish(OUT/"cache"/name, LOCAL/name)
    assert sha256(LOCAL/name) == cache_info["hashes"][name]
train_tokens = np.load(LOCAL/"train_tokens.npy", mmap_mode="r")
train_labels = np.load(LOCAL/"train_labels.npy", mmap_mode="r")
assert train_tokens.shape == (TRAIN_TOTAL, MAX_LENGTH)
assert train_labels.shape == (TRAIN_TOTAL, 2)
assert np.all(train_labels[TRAIN_SIZE:, 1] == IGNORE)
assert np.all(train_labels[TRAIN_SIZE:, 0] != IGNORE)
print("Original:", TRAIN_SIZE, "| Pseudo:", accepted_count, "| Total:", TRAIN_TOTAL)


## ۶. معیارها و نمونهٔ ثابت validation
همان روش نوت‌بوک supervised برای ساخت نمونهٔ ۱۰۰ هزار ردیفی validation استفاده می‌شود.
این نمونه فقط برای پایش و انتخاب checkpoint است؛ ارزیابی نهایی روی فایل‌های کامل انجام می‌شود.

برای هر هدف، فقط ردیف دارای برچسب معتبر در معیارها حساب می‌شود.
خطای ترتیبی rating_band میانگین فاصلهٔ پیش‌بینی و برچسب با ترتیب low=0، medium=1، high=2 است.


In [ ]:
def metrics_from_counts(cm, target):
    # معیارها از ماتریس شمارش محاسبه می‌شوند؛ نیازی به نگه‌داشتن همهٔ پیش‌بینی‌ها در RAM نیست.
    cm = np.asarray(cm, dtype=np.int64)
    support = cm.sum(axis=1)
    predicted = cm.sum(axis=0)
    tp = cm.diagonal()
    precision = np.divide(tp, predicted, out=np.zeros(3, dtype=float), where=predicted!=0)
    recall = np.divide(tp, support, out=np.zeros(3, dtype=float), where=support!=0)
    f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros(3, dtype=float), where=precision+recall!=0)
    total = int(cm.sum())
    if total == 0:
        raise ValueError("هیچ برچسب معتبری برای " + target + " وجود ندارد.")
    row = {"n":total, "Accuracy":float(tp.sum()/total), "Macro F1":float(f1.mean()),
           "Macro Precision":float(precision.mean()), "Macro Recall":float(recall.mean()),
           "Balanced Accuracy":float(recall[support>0].mean())}
    if target == "rating_band":
        distance = np.abs(np.arange(3)[:,None]-np.arange(3)[None,:])
        row["Ordinal MAE"] = float((cm*distance).sum()/total)
    per_class = pd.DataFrame({"class":TARGETS[target], "precision":precision,
                              "recall":recall, "f1":f1, "support":support})
    return row, per_class

def evaluate_frame(frame):
    predicted, _ = predict(frame)
    labels = target_array(frame)
    counts = {}
    for j,t in enumerate(TARGETS):
        valid = labels[:,j] != IGNORE
        counts[t] = confusion_matrix(labels[valid,j], predicted[t][valid], labels=[0,1,2])
    return counts

monitor_path = OUT/"cache/validation_monitor.csv"
if not monitor_path.exists():
    total = int(manifest["output_row_counts"]["validation"])
    fraction = min(1.0, MONITOR_SIZE/total*1.25)
    parts = [chunk.sample(frac=fraction, random_state=SEED+i)
             for i,chunk in enumerate(pd.read_csv(LOCAL/"validation.csv", usecols=["body", *TARGETS], chunksize=100_000))]
    frame = pd.concat(parts, ignore_index=True)
    if len(frame) > MONITOR_SIZE:
        frame = frame.sample(n=MONITOR_SIZE, random_state=SEED)
    temp = LOCAL/"monitor.csv"
    frame.to_csv(temp, index=False)
    publish(temp, monitor_path)
    del parts, frame
monitor_frame = pd.read_csv(monitor_path)
assert len(monitor_frame) == MONITOR_SIZE

def monitor_scores():
    counts = evaluate_frame(monitor_frame)
    scores = {t:metrics_from_counts(cm,t)[0] for t,cm in counts.items()}
    mean_f1 = float(np.mean([v["Macro F1"] for v in scores.values()]))
    return mean_f1, scores

initial_path = OUT/"reports/supervised_monitor.json"
if not initial_path.exists():
    model.load_state_dict(load_file(str(BASE/"model.safetensors"), device="cpu"), strict=True)
    initial_mean, initial_scores = monitor_scores()
    write_json(initial_path, {"mean_macro_f1":initial_mean,"targets":initial_scores})
display(pd.DataFrame(read_json(initial_path)["targets"]).T)


## ۷. داده‌خوان و loss دو هدف
این بخش فقط توکن‌ها را می‌خواند. padding اضافی هر batch حذف می‌شود تا محاسبه کمتر شود.
loss هر سر روی برچسب‌های معتبر همان سر حساب می‌شود؛ نمونهٔ pseudo روی rating_band هیچ loss ندارد.
قبل از آموزش یک آزمایش کوچک انجام می‌دهیم تا بارگذاری مدل و مسیر گرادیان درست باشند.


In [ ]:
class TokenDataset(Dataset):
    def __len__(self):
        return TRAIN_TOTAL

    def __getitem__(self, index):
        return (torch.from_numpy(np.array(train_tokens[index], dtype=np.int64)),
                torch.from_numpy(np.array(train_labels[index], dtype=np.int64)))

def collate_tokens(rows):
    ids = torch.stack([r[0] for r in rows])
    labels = torch.stack([r[1] for r in rows])
    mask = ids.ne(tokenizer.pad_token_id)
    length = int(mask.sum(1).max())
    return ids[:, :length], mask[:, :length], labels

def masked_loss(logits, labels):
    losses = []
    for j,t in enumerate(TARGETS):
        valid = labels[:,j] != IGNORE
        if valid.any():
            losses.append(nn.functional.cross_entropy(logits[t][valid].float(), labels[valid,j]))
    if not losses:
        raise ValueError("این batch هیچ برچسب معتبری ندارد.")
    return torch.stack(losses).sum()

dataset = TokenDataset()
model.load_state_dict(load_file(str(BASE/"model.safetensors"), device="cpu"), strict=True)
model.train()
sample_ids = [0, 1, TRAIN_SIZE, TRAIN_SIZE+min(1, accepted_count-1)]
ids, mask, labels = collate_tokens([dataset[i] for i in sample_ids])
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(ids.to(DEVICE), mask.to(DEVICE))
    loss = masked_loss(logits, labels.to(DEVICE))
assert torch.isfinite(loss)
loss.backward()
assert model.heads["recommendation_status"].weight.grad is not None
assert model.encoder.encoder.layer[-1].output.dense.weight.grad is not None
model.zero_grad(set_to_none=True)
del logits, loss, ids, mask, labels
gc.collect(); torch.cuda.empty_cache()
print("Smoke test passed؛ هنوز هیچ وزن مدل به‌روزرسانی نشده است.")


## ۸. آموزش نیمه‌نظارتی و ادامه پس از قطع اتصال
از وزن supervised شروع می‌کنیم، اما optimizer و برنامهٔ نرخ یادگیری برای ادامهٔ آموزش جدید ساخته می‌شوند.
تمام نمونه‌های اصلی و پذیرفته‌شده در یک ترتیب تصادفی ثابت مخلوط می‌شوند.
همراه وزن‌ها، وضعیت optimizer، برنامهٔ نرخ یادگیری، تعداد batchهای مصرف‌شده و وضعیت اعداد تصادفی ذخیره می‌شود.
در ادامه، batchهای کامل‌شده دوباره آموزش داده نمی‌شوند.

**اجرا:** سلول زیر را اجرا کنید. اگر پیام PAUSED دیدید، همان سلول را دوباره اجرا کنید.
اگر اتصال Colab قطع شد، از بالای نوت‌بوک شروع کنید؛ خروجی‌های ذخیره‌شده بازیابی می‌شوند.
برای توقف دستی، ممکن است کارِ بعد از آخرین checkpoint از دست برود؛ این مقدار معمولاً حداکثر ۲۰۰۰ گام است.

زمان باقیمانده پس از ۱۰۰ گام برآورد می‌شود؛ زمان انتقال فایل و ارزیابی را هم باید در نظر بگیرید.
اگر این برآورد با فرصت شما سازگار نبود، قبل از تغییر حجم آموزش یا لایه‌های قابل‌آموزش تصمیم بگیرید.
محدودیت منابع را با تغییر پنهانی آستانه یا کم‌کردن test جبران نمی‌کنیم.

هر بار که validation بهتر شود، وزن آن گام جدا ذخیره می‌شود؛ این کار مانع اشتباه‌گرفتن وزن و شمارهٔ checkpoint پس از قطع ناگهانی می‌شود.

برای اشغال‌نشدن Drive، نسخهٔ بهترین گام قبلی فقط پس از ذخیرهٔ موفق وضعیت جدید پاک می‌شود. فایل مدل supervised دست‌نخورده می‌ماند.


In [ ]:
LATEST = OUT/"checkpoints/latest_training_state.pt"
BEST = OUT/"checkpoints/best_model.safetensors"
DONE = OUT/"training_complete.json"
NUM_BATCHES = math.ceil(TRAIN_TOTAL/BATCH_SIZE)
TOTAL_STEPS = math.ceil(NUM_BATCHES/GRAD_ACCUMULATION)
order = np.random.default_rng(SEED).permutation(TRAIN_TOTAL)

def save_best(step):
    local_path = LOCAL/"best_model.safetensors"
    save_file({k:v.detach().cpu().contiguous() for k,v in model.state_dict().items()}, str(local_path))
    saved_path = OUT/"checkpoints"/f"best_step_{step}.safetensors"
    publish(local_path, saved_path)
    return saved_path.name

def train_one_session():
    if DONE.exists():
        completed = read_json(DONE)
        assert completed["run_id"] == RUN_ID and completed["best_sha256"] == sha256(BEST)
        print("COMPLETE: آموزش قبلاً تمام شده؛ بخش ارزیابی را اجرا کنید.")
        return

    optimizer = torch.optim.AdamW([
        {"params":model.encoder.parameters(), "lr":ENCODER_LR},
        {"params":model.heads.parameters(), "lr":HEAD_LR}], weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(TOTAL_STEPS*WARMUP_RATIO), TOTAL_STEPS)
    scaler = torch.amp.GradScaler("cuda")
    progress = {"next_batch":0, "step":0, "best_score":-1.0, "best_step":None,
                "best_file":None, "bad_checks":0, "loss_sum":0.0, "loss_batches":0, "skipped_updates":0,
                "train_curve":[], "validation":[], "session_seconds":0.0}

    if LATEST.exists():
        publish(LATEST, LOCAL/"resume.pt")
        saved = torch.load(LOCAL/"resume.pt", map_location="cpu", weights_only=False)
        assert saved["run_id"] == RUN_ID
        model.load_state_dict(saved["model"], strict=True)
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
        progress = saved["progress"]
        torch.set_rng_state(saved["torch_rng"])
        torch.cuda.set_rng_state_all(saved["cuda_rng"])
        random.setstate(saved["python_rng"])
        np.random.set_state(saved["numpy_rng"])
        del saved
        print("Resume from batch", progress["next_batch"], "/", NUM_BATCHES)
    else:
        model.load_state_dict(load_file(str(BASE/"model.safetensors"), device="cpu"), strict=True)
        random.seed(SEED); np.random.seed(SEED)
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    if progress["best_file"] is not None and not (OUT/"checkpoints"/progress["best_file"]).exists():
        raise FileNotFoundError("best_model مربوط به checkpoint موجود نیست؛ پوشهٔ خروجی را کامل بازیابی کنید.")

    session_start = time.time()
    previous_seconds = progress["session_seconds"]

    def save_recovery():
        progress["session_seconds"] = previous_seconds + time.time()-session_start
        state = {"run_id":RUN_ID, "model":model.state_dict(),
                 "optimizer":optimizer.state_dict(), "scheduler":scheduler.state_dict(),
                 "scaler":scaler.state_dict(), "progress":progress,
                 "torch_rng":torch.get_rng_state(), "cuda_rng":torch.cuda.get_rng_state_all(),
                 "python_rng":random.getstate(), "numpy_rng":np.random.get_state()}
        torch.save(state, LOCAL/"training_state.pt")
        publish(LOCAL/"training_state.pt", LATEST)
        write_json(OUT/"reports/training_history.json", progress)
        # فقط پس از ذخیرهٔ موفق recovery، وزن‌های بهترین گام‌های قدیمی حذف می‌شوند.
        for old_best in (OUT/"checkpoints").glob("best_step_*.safetensors"):
            if old_best.name != progress["best_file"]:
                old_best.unlink()

    start_batch = progress["next_batch"]
    # generator جدا مانع تغییر وضعیت تصادفی dropout هنگام ساخت DataLoader می‌شود.
    loader = DataLoader(Subset(dataset, order[start_batch*BATCH_SIZE:]),
                        batch_size=BATCH_SIZE, num_workers=0, collate_fn=collate_tokens,
                        pin_memory=True, generator=torch.Generator().manual_seed(SEED))
    optimizer.zero_grad(set_to_none=True)
    pending = 0
    session_steps = 0
    finished = progress["next_batch"] >= NUM_BATCHES or progress["bad_checks"] >= PATIENCE
    reason = "early_stopping" if progress["bad_checks"] >= PATIENCE else "one_epoch_completed"
    model.train()
    bar = tqdm(loader, total=NUM_BATCHES-start_batch, desc="Semi-supervised training")
    if not finished:
        for local_batch, (ids, attention, labels) in enumerate(bar):
            batch_number = start_batch + local_batch + 1
            ids, attention, labels = ids.to(DEVICE), attention.to(DEVICE), labels.to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(ids, attention)
                loss = masked_loss(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError("loss نامعتبر است؛ آخرین checkpoint سالم حفظ شده است.")
            scaler.scale(loss/GRAD_ACCUMULATION).backward()
            pending += 1
            progress["loss_sum"] += float(loss.detach().cpu())
            progress["loss_batches"] += 1
            if pending < GRAD_ACCUMULATION and batch_number < NUM_BATCHES:
                continue

            # آخرین گروه ممکن است کمتر از چهار batch داشته باشد.
            if pending < GRAD_ACCUMULATION:
                for parameter in model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.mul_(GRAD_ACCUMULATION/pending)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= old_scale:
                scheduler.step()
            else:
                progress["skipped_updates"] += 1
            optimizer.zero_grad(set_to_none=True)
            pending = 0
            progress["step"] += 1
            progress["next_batch"] = batch_number
            session_steps += 1
            step = progress["step"]
            mean_loss = progress["loss_sum"]/progress["loss_batches"]
            bar.set_postfix(loss=f"{mean_loss:.4f}", step=step)

            if step % 200 == 0 or batch_number == NUM_BATCHES:
                progress["train_curve"].append({"step":step, "loss":mean_loss})
            if session_steps == 100:
                seconds_per_step = (time.time()-session_start)/session_steps
                hours = seconds_per_step*(TOTAL_STEPS-step)/3600
                print(f"Estimated remaining training: {hours:.2f} hours; evaluation and saves add time.")

            evaluated = step % EVAL_EVERY == 0 or batch_number == NUM_BATCHES
            if evaluated:
                score, targets = monitor_scores()
                progress["validation"].append({"step":step, "mean_macro_f1":score, "targets":targets})
                print("Validation mean Macro F1:", round(score, 4))
                if score > progress["best_score"]:
                    progress["best_file"] = save_best(step)
                    progress["best_score"], progress["best_step"], progress["bad_checks"] = score, step, 0
                else:
                    progress["bad_checks"] += 1
                model.train()

            finished = batch_number == NUM_BATCHES or progress["bad_checks"] >= PATIENCE
            reason = "early_stopping" if progress["bad_checks"] >= PATIENCE else "one_epoch_completed"
            pause = time.time()-session_start >= SESSION_MINUTES*60
            if step % SAVE_EVERY == 0 or evaluated or finished or pause:
                save_recovery()
            if finished or pause:
                break
    bar.close()

    if finished:
        assert progress["best_file"] is not None and progress["best_step"] is not None
        save_recovery()
        publish(OUT/"checkpoints"/progress["best_file"], BEST)
        write_json(DONE, {"run_id":RUN_ID, "reason":reason, "completed_batches":progress["next_batch"],
                          "total_batches":NUM_BATCHES, "best_step":progress["best_step"],
                          "best_validation_mean_macro_f1":progress["best_score"],
                          "best_sha256":sha256(BEST), "finished_unix":time.time()})
        print("COMPLETE: آمادهٔ ارزیابی نهایی است.")
    else:
        print("PAUSED: ذخیره شد. برای ادامه، همین سلول را دوباره اجرا کنید.")
    del loader, optimizer, scheduler, scaler
    gc.collect(); torch.cuda.empty_cache()

train_one_session()


## ۹. نمودار آموزش و ذخیرهٔ مدل انتخاب‌شده
اگر آموزش فقط مکث کرده باشد، این سلول اجازهٔ ادامه نمی‌دهد.
نمودار loss میانگین تجمعیِ batchهای آموزش است؛ نمودار validation امتیاز Macro F1 هر سر را نشان می‌دهد.
نقطهٔ صفر، مدل supervised اولیه است. انتخاب checkpoint فقط بین مدل‌های نیمه‌نظارتی انجام شده است.


In [ ]:
if not DONE.exists():
    raise RuntimeError("آموزش هنوز تمام نشده؛ سلول قبل را ادامه دهید. PAUSED با COMPLETE فرق دارد.")
completion = read_json(DONE)
assert completion["run_id"] == RUN_ID and completion["best_sha256"] == sha256(BEST)
history = read_json(OUT/"reports/training_history.json")
fig, axes = plt.subplots(1,2,figsize=(13,4))
curve = history["train_curve"]
axes[0].plot([r["step"] for r in curve], [r["loss"] for r in curve])
axes[0].set(title="Cumulative training loss", xlabel="Update step", ylabel="Loss")
initial = read_json(OUT/"reports/supervised_monitor.json")
for target in TARGETS:
    x = [0] + [r["step"] for r in history["validation"]]
    y = [initial["targets"][target]["Macro F1"]] + [r["targets"][target]["Macro F1"] for r in history["validation"]]
    axes[1].plot(x,y,marker="o",label=target)
axes[1].axvline(completion["best_step"], color="grey", linestyle="--", label="Selected checkpoint")
axes[1].set(title="Validation monitor", xlabel="Update step", ylabel="Macro F1")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT/"reports/training_curves.png", dpi=150)
plt.show(); plt.close(fig)

# فایل supervised اصلی هیچ‌وقت بازنویسی نمی‌شود.
publish(BEST, OUT/"artifacts/model.safetensors")
tokenizer.save_pretrained(str(OUT/"artifacts/tokenizer"))
config.save_pretrained(str(OUT/"artifacts/encoder_config"))
write_json(OUT/"artifacts/model_metadata.json", {
    "base_checkpoint":metadata["base_checkpoint"], "input":"body", "targets":TARGETS,
    "max_length":MAX_LENGTH, "rating_band_thresholds":manifest["rating_band_thresholds"],
    "training_config":settings, "supervised_weight_sha256":fingerprints["model.safetensors"],
    "selected_checkpoint":completion, "total_parameters":total_params, "trainable_parameters":trainable_params,
    "weight_file":"model.safetensors", "tokenizer_dir":"tokenizer", "encoder_config_dir":"encoder_config"})
print("Selected step:", completion["best_step"])


## ۱۰. ارزیابی کامل supervised و semi-supervised
هر دو مدل را روی **همان فایل‌های کامل validation و test** اجرا می‌کنیم.
جدول قدیمی supervised را مستقیماً استفاده نمی‌کنیم، چون برای آن هش ورودی و همهٔ معیارهای لازم ثبت نشده بود.
این بار هویت فایل‌ها برای هر دو مدل مشخص است.

ارزیابی هر ۲۰ هزار ردیف ذخیره می‌شود؛ پس از قطع اتصال، بخش‌های تمام‌شده تکرار نمی‌شوند.
پیش‌بینی هر ردیف هم در فایل‌های کوچک npz باقی می‌ماند تا بعداً برای تحلیل خطا نیاز به اجرای دوبارهٔ GPU نباشد.
این بخش آموزش انجام نمی‌دهد. بعد از دیدن test تنظیمات مدل را تغییر ندهید.


In [ ]:
def full_evaluation(model_name, weights):
    if not DONE.exists():
        raise RuntimeError("ابتدا آموزش را تمام کنید.")
    weight_hash = sha256(weights)
    expected = fingerprints["model.safetensors"] if model_name == "supervised" else read_json(DONE)["best_sha256"]
    assert weight_hash == expected
    model.load_state_dict(load_file(str(weights), device="cpu"), strict=True)
    model.eval()
    rows = []
    for split in ["validation", "test"]:
        folder = OUT/"evaluation"/model_name/split
        folder.mkdir(parents=True, exist_ok=True)
        identity = {"run_id":RUN_ID, "weights_sha256":weight_hash,
                    "input_sha256":fingerprints[split+".csv"], "chunk_rows":CHUNK_ROWS}
        if (folder/"identity.json").exists():
            assert read_json(folder/"identity.json") == identity
        else:
            write_json(folder/"identity.json", identity)
        totals = {t:np.zeros((3,3), dtype=np.int64) for t in TARGETS}
        offset = 0
        reader = pd.read_csv(LOCAL/(split+".csv"), usecols=["body", *TARGETS], chunksize=CHUNK_ROWS)
        for part, frame in enumerate(tqdm(reader, desc=model_name+" "+split)):
            dest = folder/f"part_{part:05d}.npz"
            if dest.exists():
                with np.load(dest, allow_pickle=False) as cached:
                    pred = cached["pred"].copy()
                    assert int(cached["offset"]) == offset
            else:
                predicted, _ = predict(frame)
                pred = np.column_stack([predicted[t] for t in TARGETS]).astype(np.int8)
                temp = LOCAL/"evaluation_part.npz"
                np.savez_compressed(temp, offset=np.asarray(offset), pred=pred)
                publish(temp, dest)
            assert pred.shape == (len(frame),2) and np.isin(pred,[0,1,2]).all()
            labels = target_array(frame)
            for j,t in enumerate(TARGETS):
                valid = labels[:,j] != IGNORE
                totals[t] += confusion_matrix(labels[valid,j], pred[valid,j], labels=[0,1,2])
            offset += len(frame)
        assert offset == manifest["output_row_counts"][split]
        fig, axes = plt.subplots(1,2,figsize=(12,4))
        for j,(target,cm) in enumerate(totals.items()):
            metrics, classes = metrics_from_counts(cm,target)
            rows.append({"model":model_name, "split":split, "target":target, **metrics})
            prefix = OUT/"reports"/f"{model_name}_{split}_{target}"
            classes.to_csv(str(prefix)+"_per_class.csv", index=False)
            pd.DataFrame(cm,index=TARGETS[target],columns=TARGETS[target]).to_csv(str(prefix)+"_confusion.csv")
            ax = axes[j]
            ax.imshow(cm, cmap="Blues")
            ax.set(xticks=range(3),yticks=range(3),xticklabels=TARGETS[target],yticklabels=TARGETS[target],
                   xlabel="Predicted",ylabel="True",title=target)
            ax.tick_params(axis="x", rotation=20)
            for a in range(3):
                for b in range(3):
                    ax.text(b,a,str(cm[a,b]),ha="center",va="center",
                            color="white" if cm[a,b]>cm.max()/2 else "black")
        fig.suptitle(model_name+" / "+split)
        fig.tight_layout()
        fig.savefig(OUT/"reports"/f"{model_name}_{split}_confusion.png",dpi=150)
        plt.show(); plt.close(fig)
    result = pd.DataFrame(rows)
    result.to_csv(OUT/"reports"/f"{model_name}_metrics.csv",index=False)
    return result

supervised_metrics = full_evaluation("supervised", BASE/"model.safetensors")
semi_metrics = full_evaluation("semi_supervised", BEST)


## ۱۱. مقایسهٔ نتیجه‌ها
delta برابر نیمه‌نظارتی منهای supervised است.
برای Accuracy و F1 و Precision و Recall و Balanced Accuracy عدد مثبت بهتر است.
برای Ordinal MAE عدد **منفی** بهتر است.
n تعداد ردیف‌های دارای برچسب معتبر همان هدف است؛ ممکن است با تعداد کل ردیف‌های split تفاوت داشته باشد.

اگر مدل دوم بهتر شد، فقط می‌توان گفت «این ادامهٔ آموزش همراه با pseudo-label بهتر بود».
در این آزمایش، اثر آموزش بیشتر از اثر دادهٔ pseudo جدا نشده است.
برای جداکردن آن‌ها یک اجرای کنترلیِ اضافه لازم است که به‌خاطر محدودیت زمان فعلاً انجام نمی‌دهیم.


In [ ]:
keys = ["split","target"]
comparison = supervised_metrics.drop(columns="model").merge(
    semi_metrics.drop(columns="model"), on=keys, suffixes=("_supervised","_semi"), validate="one_to_one")
assert comparison["n_supervised"].eq(comparison["n_semi"]).all()
metric_names = ["Accuracy","Macro F1","Macro Precision","Macro Recall","Balanced Accuracy","Ordinal MAE"]
for name in metric_names:
    comparison[name+"_delta"] = comparison[name+"_semi"] - comparison[name+"_supervised"]
comparison.to_csv(OUT/"reports/comparison.csv", index=False)
display(comparison.round(4))
for _,row in comparison.iterrows():
    print(row["split"], row["target"], "| Macro F1 change:", f"{100*row['Macro F1_delta']:+.2f} percentage points")
write_json(OUT/"reports/evaluation_protocol.json", {
    "run_id":RUN_ID, "inputs":fingerprints, "selection":"validation monitor only; no test selection",
    "supervised_model_sha256":fingerprints["model.safetensors"],
    "semi_supervised_model_sha256":read_json(DONE)["best_sha256"],
    "evaluation":"full validation and test; independent masks per target",
    "limitations":["Different additional training exposure; no extra supervised-only continuation control.",
                   "Historical supervised input hashes are unavailable; same subset reconstructed from recorded seed and row indices.",
                   "Floating point results can vary slightly between GPU types and library versions."]})


## ۱۲. حالا برچسب‌های مخفی را فقط برای سنجش کیفیت بخوانیم
این بخش **بعد از انتخاب pseudo-label و پایان آموزش** اجرا می‌شود.
mask_manifest.csv در preprocessing هم‌زمان و به همان ترتیب فایل مصنوعی ساخته شده است.
ابتدا تعداد و ترتیب idها را کنترل می‌کنیم، بعد فقط برچسب‌های پذیرفته‌شده را با مرجع مقایسه می‌کنیم.
این روش id تکراری را به چند ردیف اشتباه join نمی‌کند.

این نتیجه برای گزارش است؛ نباید با دیدن آن آستانهٔ ۰٫۹۰ یا انتخاب pseudo-labelها را تغییر دهید.
درستی pseudo-label و درستی مدل روی test دو مفهوم جدا هستند.


In [ ]:
if not DONE.exists():
    raise RuntimeError("خواندن مرجع تا پایان آموزش مجاز نیست.")
completion = read_json(DONE)
assert completion["run_id"] == RUN_ID and completion["best_sha256"] == sha256(BEST)
selection = read_json(OUT/"reports/pseudo_selection.json")
assert selection["threshold"] == 0.90
prediction_parts = []
for name,digest in selection["prediction_hashes"].items():
    source = OUT/"pseudo"/name
    assert sha256(source) == digest
    prediction_parts.append(pd.read_csv(source,dtype={"id":str},keep_default_na=False))
predictions = pd.concat(prediction_parts,ignore_index=True)
assert np.array_equal(predictions["pool_row"].to_numpy(), np.arange(len(predictions)))

# اولین دسترسی به محتوای برچسب‌های واقعی مخفی‌شده در همین نقطه است.
reference_path = DATA/"mask_manifest.csv"
reference = pd.read_csv(reference_path,usecols=["id","recommendation_status"],dtype=str,keep_default_na=False)
if len(reference) != len(predictions) or not np.array_equal(reference["id"].to_numpy(),predictions["id"].to_numpy()):
    raise ValueError("تعداد یا ترتیب مرجع با فایل مصنوعی یکسان نیست؛ audit متوقف شد. مرجع را بررسی کنید.")
true_ids = reference["recommendation_status"].map(LABEL_TO_ID["recommendation_status"])
assert true_ids.notna().all()
keep = predictions["confidence"].ge(THRESHOLD).to_numpy()
assert int(keep.sum()) == selection["accepted_rows"]
y = true_ids.to_numpy(dtype=np.int64)[keep]
p = predictions.loc[keep,"prediction"].to_numpy(dtype=np.int64)
cm = confusion_matrix(y,p,labels=[0,1,2])
audit_metrics, audit_classes = metrics_from_counts(cm,"recommendation_status")
audit_classes.to_csv(OUT/"reports/pseudo_quality_per_class.csv",index=False)
pd.DataFrame(cm,index=TARGETS["recommendation_status"],columns=TARGETS["recommendation_status"]).to_csv(
    OUT/"reports/pseudo_quality_confusion.csv")
audit = {"accepted_rows":int(keep.sum()),"pool_rows":len(predictions),
         "coverage_percent":selection["accepted_percent"], "accepted_class_counts":selection["class_counts"],
         "accepted_pseudo_label_accuracy":float((y==p).mean()),
         "reference_sha256":sha256(reference_path),"audit_unix":time.time(),
         "training_finished_unix":completion["finished_unix"],"metrics":audit_metrics}
write_json(OUT/"reports/pseudo_quality.json",audit)
print("Accepted pseudo-label accuracy:", round(audit["accepted_pseudo_label_accuracy"],4))
display(audit_classes.round(4))
del prediction_parts, predictions, reference
gc.collect()


## ۱۳. خروجی‌ها و برنامهٔ پیشنهادی برای چهار روز
همهٔ نتیجه‌ها در **outputs/semi_supervised_v1** هستند:

- **artifacts/**: مدل انتخاب‌شده، tokenizer و مشخصات مدل.
- **reports/comparison.csv**: مقایسهٔ supervised و نیمه‌نظارتی.
- **reports/pseudo_selection.json**: تعداد پذیرفته‌ها، درصد پوشش و توزیع کلاس‌ها.
- **reports/pseudo_quality.json**: دقت pseudo-labelها پس از پایان آموزش.
- **reports/**: معیارهای هر کلاس، ماتریس‌ها، نمودار آموزش و جزئیات اجرا.
- **evaluation/**: پیش‌بینی‌های هر بخش؛ برای تحلیل خطای مرحلهٔ بعد نگه دارید.
- **checkpoints/**: وضعیت ادامهٔ آموزش و وزن انتخاب‌شده؛ تا پایان کار حذف نکنید.

**پیشنهاد زمان‌بندی:** روز اول آماده‌سازی و pseudo-label؛ روز دوم ادامهٔ آموزش؛
روز سوم ارزیابی کامل و audit؛ روز چهارم تحلیل خطاها و تکمیل گزارش.
این تقسیم‌بندی تضمین زمان نیست؛ سرعت GPU، دسترسی به سهمیه و انتقال Drive تعیین‌کننده‌اند.
پس از برآورد زمان در شروع آموزش، برنامه را با زمان واقعی هماهنگ کنید.

اگر فقط مکث شده: سلول آموزش را دوباره اجرا کنید.
اگر runtime عوض شده: همهٔ سلول‌ها را از ابتدا و به ترتیب اجرا کنید.
اگر در ارزیابی قطع شده: بخش‌های کامل از Drive خوانده می‌شوند و باقی ادامه پیدا می‌کند.
اگر خطای کمبود حافظه در inference رخ داد: پیش از اجرای اصلی batch استنتاج را کوچک‌تر و OUT را جدید کنید؛
در میانهٔ آزمایش تنظیمات را بی‌ثبت تغییر ندهید.

**محدودهٔ اعتبارسنجی:** این فایل محلی با آزمون‌های کوچک بررسی شده است.
اجرای کامل T4 و چندمیلیونی باید در Colab انجام شود؛ زمان یا نتیجهٔ نهایی از قبل ادعا نمی‌شود.
راهنمای رسمی: Google Colab FAQ؛ PyTorch Automatic Mixed Precision recipe؛
Hugging Face Transformers Auto Classes.
